# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

d:\AI_Projects\llm_engineering\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://huggingface.co")
links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/storage',
 '/docs',
 '/enterprise',
 '/pricing',
 '/tasks',
 '/chat',
 '/collections',
 '/languages',
 '/organizations',
 '/blog/zh',
 '/posts',
 '/papers',
 '/hardware',
 '/learn',
 '/join/discord',
 'https://discuss.huggingface.co/',
 'https://github.com/huggingface',
 '/enterprise',
 '/pro',
 '/support',
 '/inference/models',
 '/inference-endpoints',
 '/storage',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic',
 '/MiniMaxAI/MiniMax-H3-Image',
 '/Qwen/Qwen3.8-27B',
 '/Lightricks/LTX-2.5',
 '/Comfy-Org/MiniMax-H3-Image',
 '/models',
 '/spaces/huggingface/moe-blogpost',
 '/spaces/joelniklaus/harness-leaderboard',
 '/spaces/kulkas2pintu/wan777',
 '/spaces/MiniMaxAI/minimax-h3-image',
 '/spaces/JCisHERE/wan2-2-i2v-v3',
 '/spaces',
 '/datasets/Angais/pixel-art-atlas-1.4m',
 '/datasets/Vyber07/cyber-security',
 '/datasets/nvidia/simready-assets',
 '/datasets/tencent/TailorRig',
 '/datase

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://huggingface.co"))


Here is the list of links on the website https://huggingface.co -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/
/models
/datasets
/spaces
/storage
/docs
/enterprise
/pricing
/tasks
/chat
/collections
/languages
/organizations
/blog/zh
/posts
/papers
/hardware
/learn
/join/discord
https://discuss.huggingface.co/
https://github.com/huggingface
/enterprise
/pro
/support
/inference/models
/inference-endpoints
/storage
/login
/join
/spaces
/models
/xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic
/MiniMaxAI/MiniMax-H3-Image
/Qwen/Qwen3.8-27B
/Lightricks/LTX-2.5
/Comfy-Org/MiniMax-H3-Image
/models
/spaces/huggingface/moe-blogpost
/spaces/joelniklaus/harness-leaderboard
/spaces/kulkas2pintu/wan777
/spaces/MiniMaxAI/minimax-h3-image
/spaces/JCisHERE/wan2-2-i2v-v3
/spaces
/datasets/Angais/pixel-art-atlas

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://huggingface.co")

{'links': [{'type': 'homepage', 'url': 'https://huggingface.co'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'docs transformers page',
   'url': 'https://huggingface.co/docs/transformers'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Status page', 'url': 'https://status.huggingface.co/'}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 12 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'resume page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'portfolio project',
   'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio project', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [ ]:
select_relevant_links("https://huggingface.co")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [10]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [11]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Search models, datasets, users...
Ctrl K
Models
Datasets
Spaces
Buckets
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic
Updated
4 days ago
•
29.6k
•
7.62k
MiniMaxAI/MiniMax-H3-Image
Updated
about 1 hour ago
•
609k
•
1.24k
Qwen/Qwen3.8-27B
Updated
9 days ago
•
6.76M
•
20.2k
Lightricks/LTX-2.5
Updated
Oct 11
•
2.

In [12]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [13]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [14]:
get_brochure_user_prompt("HUGGING FACE", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 4 relevant links


'\nYou are looking at a company called: HUGGING FACE\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nSearch models, datasets, users...\nCtrl K\nModels\nDatasets\nSpaces\nBuckets\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nxtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic\nUpdated\n4 days ago\

In [15]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [16]:
create_brochure("HUGGING FACE", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the pioneering AI community and collaboration platform that is building the future of machine learning. As the home of machine learning innovation, it serves as a central hub where engineers, researchers, and scientists across the globe can share, explore, and experiment with over 2 million models, 500,000 datasets, and 1 million+ AI applications. Hugging Face empowers the next generation of machine learning professionals and end-users to contribute to and shape an open, ethical, and collaborative AI ecosystem.

---

## What We Offer

- **Models & Datasets:** Discover and contribute to a vast collection of cutting-edge AI models and datasets, ranging from natural language processing to computer vision.
- **Spaces:** Host and collaborate on machine learning applications and demonstrations seamlessly with our interactive "Spaces" platform.
- **Tools & Libraries:** Access some of the most widely-used open-source ML libraries and tools in the industry.
- **Enterprise Solutions:** Specialized plans for teams and organizations, including Hugging Face PRO, enterprise support, inference endpoints, and storage buckets tailored to scale AI deployments securely and efficiently.
- **Learning & Community:** Engage with an active global community through forums, Discord, blogs, and shared resources that support continuous education and collaboration.

---

## Company Culture

- **Community-Driven:** Hugging Face thrives on open collaboration and transparency, encouraging users and contributors to share knowledge freely.
- **Innovation at the Edge:** Our talented science and engineering teams explore the latest in AI technology, pushing the boundaries of what's possible.
- **Ethical AI:** Committed to building a responsible AI future, Hugging Face fosters openness and ethical practices in AI development and deployment.
- **Inclusivity:** Welcoming people worldwide, empowering diverse voices in AI research and application.

---

## Customers & Users

Hugging Face serves a wide range of users and customers, including:

- AI researchers and machine learning engineers seeking state-of-the-art open-source resources.
- Enterprises and startups deploying custom ML solutions with scalable infrastructure.
- Educators and students using interactive tools to learn and experiment with AI.
- Developers building innovative applications powered by machine learning.

---

## Careers at Hugging Face

Join a fast-growing company at the heart of the AI revolution, working alongside brilliant minds passionate about shaping the future through technology.

- Open roles span software engineering, machine learning research, product management, and more.
- Work culture values collaboration, transparency, and continuous learning.
- Opportunities to contribute to impactful open-source projects and lead innovation.
- Supportive environment promoting diversity, inclusivity, and ethical AI development.

Explore career opportunities and apply at [huggingface.co/careers](https://huggingface.co/careers).

---

## Get Involved

- Explore and use models at [huggingface.co/models](https://huggingface.co/models)
- Access datasets: [huggingface.co/datasets](https://huggingface.co/datasets)
- Join the community: Discord, GitHub, Forums.
- Discover brand assets and resources for collaboration and projects.

---

## Contact & Social

- Website: [huggingface.co](https://huggingface.co)
- GitHub: github.com/huggingface
- Twitter: @huggingface
- LinkedIn: Hugging Face
- Discord: Community Server

---

**Hugging Face — Powering the AI community to collaborate, innovate, and build the future of machine learning together.**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [18]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [19]:
stream_brochure("HUGGING FACE", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the leading collaboration platform for the machine learning (ML) community, dedicated to building the future of AI together. At its core, Hugging Face empowers machine learning engineers, scientists, and end users to learn, collaborate, share, and innovate through an open and ethical AI ecosystem.

The Hugging Face Hub serves as a central place where over 2 million models, 500,000 datasets, and 1 million+ applications are created, shared, explored, and experimented with every day. By fostering a fast-growing community and providing some of the most widely used open-source ML libraries and tools, Hugging Face is at the heart of the AI revolution.

---

## Company Culture

- **Open Collaboration**: Hugging Face believes in an open, transparent, and ethical approach to AI development, supporting a rich community that contributes models, datasets, and applications.
- **Innovation Driven**: The company is powered by a talented science team constantly exploring the edge of machine learning technology.
- **Community Focused**: The platform thrives on collaboration and engagement through forums, Discord, GitHub, and educational resources.
- **Inclusivity & Learning**: Whether you are a novice or an expert, Hugging Face provides extensive documentation, guides, and a welcoming community for continuous learning and growth.
- **Ethical AI Advocacy**: Hugging Face promotes building AI solutions responsibly to benefit and enhance society.

---

## Solutions and Offerings

- **Hugging Face Hub**  
  A central platform to share over 2 million pre-trained models and 500,000+ datasets, covering diverse ML tasks from NLP, computer vision, to audio and reinforcement learning.

- **Spaces**  
  Host and explore end-user AI applications, facilitating hands-on AI experiences powered by community and enterprise developers.

- **Enterprise Solutions**
  - **Team & Enterprise Plans** designed to support organizational collaboration with features like private model hosting, dedicated infrastructure, and security.
  - **Hugging Face PRO** for advanced enterprise support.
  - **Inference Providers & Endpoints** to deploy and scale ML models efficiently in production.
  - **Storage Buckets** to securely manage data and models.

---

## Customers & Community

Hugging Face serves a wide range of users globally including:

- Independent machine learning engineers and researchers.
- AI-first startups and technology companies.
- Large enterprises requiring ML model management and deployment solutions.
- Academic institutions fostering AI research and education.

The vibrant community contributes not only models and datasets but also participates actively in forums, Discord channels, daily research discussions, and open challenges.

---

## Careers & Opportunities

Hugging Face thrives on attracting passionate individuals eager to:

- Advance the state-of-the-art in machine learning.
- Build scalable AI tools and open-source libraries.
- Engage and grow a global AI community.
- Drive ethical and responsible AI development.

Join a team at the forefront of AI innovation where your work will impact millions globally. Hugging Face offers roles in ML research, software engineering, community management, enterprise support, and more.

Find out about current openings and apply via their [Careers page](https://huggingface.co/careers).

---

## Branding & Visual Identity

Hugging Face maintains a vibrant and recognizable brand identity featuring:

- Signature colors: bright yellow (#FFD21E), bold orange (#FF9D00), and neutral gray (#6B7280).
- Friendly logo assets available in multiple formats (.svg, .png, .ai) to promote community and partner use.

---

## Connect with Hugging Face

- Website: [huggingface.co](https://huggingface.co)
- GitHub: github.com/huggingface
- Discord & Forum: For community engagement and support.
- Social Media: Twitter & LinkedIn for updates and announcements.

---

**Hugging Face**  
*The AI community building the future.*

Explore the future of AI with Hugging Face — where collaboration, innovation, and open science meet!

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>